In [1]:
import numpy as np

class GridWorld:
    def __init__(self, width, height, obstacle_density=0.2, obstacle_mode='random', seed=None,
                 cluster_size=5, start=None, goal=None, allow_diagonal=False, diagonal_cost=None,
                 reward_goal=100, reward_obstacle=-10, reward_step=-1, shaping=None):
        """
        GridWorld environment for Q-learning.
        - width, height: grid dimensions.
        - obstacle_density: fraction of cells that are obstacles (0.0 to 1.0).
        - obstacle_mode: 'random' for uniform random obstacles, or 'cluster' for clustered obstacles.
        - seed: random seed for reproducible obstacle placement.
        - cluster_size: approximate cluster size (number of obstacle cells per cluster) if using clustered mode.
        - start, goal: (row, col) tuples for start and goal positions. If None, defaults to (0,0) and (height-1, width-1).
        - allow_diagonal: whether 8-directional movement is allowed (if False, only 4-directional moves).
        - diagonal_cost: cost factor for diagonal moves (if None, uses sqrt(2) for diagonal vs 1 for straight moves by default).
        - reward_goal: reward for reaching the goal.
        - reward_obstacle: reward (penalty, typically negative) for hitting an obstacle or wall (invalid move).
        - reward_step: reward (often negative or 0) for a normal step (to encourage shorter paths, use a small negative).
        - shaping: None (no reward shaping) or 'manhattan' or 'euclidean' to add potential-based reward shaping.
        """
        self.width = width
        self.height = height
        self.obstacle_density = obstacle_density
        self.obstacle_mode = obstacle_mode
        self.seed = seed
        self.cluster_size = cluster_size
        self.allow_diagonal = allow_diagonal
        # Diagonal movement cost (if not provided, default to sqrt(2) for diagonal, 1 for straight)
        if diagonal_cost is None:
            self.diagonal_cost = np.sqrt(2) if allow_diagonal else 1.0
        else:
            self.diagonal_cost = diagonal_cost
        # Rewards configuration
        self.reward_goal = reward_goal
        self.reward_obstacle = reward_obstacle
        self.reward_step = reward_step
        self.shaping = shaping  # reward shaping metric if any
        # Initialize random generator
        if seed is not None:
            self.rng = np.random.RandomState(seed)
        else:
            self.rng = np.random
        # Create grid and populate obstacles
        self.grid = np.zeros((height, width), dtype=np.int8)  # 0=free, 1=obstacle
        self._generate_obstacles()  # fill self.grid with obstacles according to mode and density
        # Define start and goal positions
        if start is None:
            self.start = (0, 0)
        else:
            self.start = tuple(start)
        if goal is None:
            self.goal = (height - 1, width - 1)
        else:
            self.goal = tuple(goal)
        # Ensure start and goal cells are free
        self.grid[self.start] = 0
        self.grid[self.goal] = 0
        # Define action set (movements) based on allow_diagonal
        if allow_diagonal:
            # 8 possible moves: N, S, W, E, NW, NE, SW, SE (row, col offsets)
            self.actions = [(-1, 0), (1, 0), (0, -1), (0, 1),
                            (-1, -1), (-1, 1), (1, -1), (1, 1)]
        else:
            # 4 possible moves: up, down, left, right
            self.actions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        self.num_actions = len(self.actions)
        # Agent's current position (set in reset())
        self.agent_position = None

    def _generate_obstacles(self):
        """Populate the grid with obstacles (1s) according to selected mode and density."""
        total_cells = self.width * self.height
        num_obstacles = int(self.obstacle_density * total_cells)
        # Start with all cells free
        self.grid.fill(0)
        if num_obstacles <= 0:
            return  # no obstacles to place
        if self.obstacle_mode == 'random':
            # Uniform random selection of obstacle cells
            obstacle_indices = self.rng.choice(total_cells, size=num_obstacles, replace=False)
            for idx in obstacle_indices:
                r = idx // self.width
                c = idx % self.width
                self.grid[r, c] = 1
        elif self.obstacle_mode == 'cluster':
            # Clustered obstacle placement: pick cluster centers and fill around them
            placed = 0
            cluster_count = max(1, num_obstacles // max(1, self.cluster_size))
            for _ in range(cluster_count):
                if placed >= num_obstacles:
                    break
                # Choose a random cluster center
                center_r = self.rng.randint(0, self.height)
                center_c = self.rng.randint(0, self.width)
                # Place obstacles around the center within a radius
                cluster_cells = min(self.cluster_size, num_obstacles - placed)
                for i in range(cluster_cells):
                    # Random offset around center
                    dr = self.rng.randint(-self.cluster_size, self.cluster_size + 1)
                    dc = self.rng.randint(-self.cluster_size, self.cluster_size + 1)
                    rr, cc = center_r + dr, center_c + dc
                    if 0 <= rr < self.height and 0 <= cc < self.width and self.grid[rr, cc] == 0:
                        self.grid[rr, cc] = 1
                        placed += 1
                        if placed >= num_obstacles:
                            break
                # If cluster placed fully or ran out of obstacles, continue to next
            # If we haven't placed all obstacles (due to overlaps), place the remainder randomly
            if placed < num_obstacles:
                free_positions = np.transpose(np.nonzero(self.grid == 0))
                remaining = num_obstacles - placed
                if len(free_positions) > 0:
                    chosen_idx = self.rng.choice(len(free_positions), size=min(remaining, len(free_positions)), replace=False)
                    for idx in chosen_idx:
                        r, c = free_positions[idx]
                        self.grid[r, c] = 1
                        placed += 1
        elif self.obstacle_mode == 'maze':
            # Placeholder for maze generation logic (not fully implemented in this prototype).
            # For now, default to random obstacles if 'maze' mode is chosen.
            obstacle_indices = self.rng.choice(total_cells, size=num_obstacles, replace=False)
            for idx in obstacle_indices:
                r = idx // self.width
                c = idx % self.width
                self.grid[r, c] = 1

    def reset(self):
        """Reset the agent to the start position and return the initial state."""
        self.agent_position = self.start
        return self.start

    def step(self, action):
        """
        Execute one action in the environment.
        :param action: index of the action in the self.actions list.
        :return: (next_state, reward, done) 
                 where next_state is a (row, col) tuple, 
                 reward is a float, 
                 done is True if the episode ended (reached goal).
        """
        if self.agent_position is None:
            # If environment not reset yet, start at the beginning
            self.agent_position = self.start
        r, c = self.agent_position
        dr, dc = self.actions[action]        # action is an index into the movement list
        nr, nc = r + dr, c + dc             # candidate next position
        # Check for validity of the move
        if nr < 0 or nr >= self.height or nc < 0 or nc >= self.width or self.grid[nr, nc] == 1:
            # Next position is out of bounds or an obstacle: invalid move
            next_state = (r, c)  # stay in the same place
            reward = self.reward_obstacle   # apply obstacle/wall penalty
            done = False                    # episode continues (hitting a wall doesn't end the episode)
        else:
            # Valid move
            next_state = (nr, nc)
            self.agent_position = next_state
            if next_state == self.goal:
                # Reached the goal
                reward = self.reward_goal
                done = True
            else:
                # Not at goal yet, apply step cost
                if self.allow_diagonal and (dr != 0 and dc != 0):
                    # Diagonal move
                    # If using negative step rewards, apply a larger negative for diagonal (approx sqrt(2) steps)
                    if self.reward_step < 0:
                        reward = -abs(self.diagonal_cost)  # e.g., -1.414 if diagonal_cost=1.414
                    else:
                        # If reward_step is positive (unusual), we could scale it similarly (not common in navigation tasks)
                        reward = self.reward_step
                else:
                    # Straight move (or no diagonal allowed)
                    reward = self.reward_step
                done = False
        # Add potential-based reward shaping if enabled
        if self.shaping is not None:
            # Compute the "potential" (distance to goal) before and after the move
            if self.shaping == 'manhattan':
                # Manhattan distance (L1) to goal
                curr_dist = abs(r - self.goal[0]) + abs(c - self.goal[1])
                new_dist = abs(next_state[0] - self.goal[0]) + abs(next_state[1] - self.goal[1])
            elif self.shaping == 'euclidean':
                # Euclidean distance (L2) to goal
                curr_dist = np.hypot(r - self.goal[0], c - self.goal[1])
                new_dist = np.hypot(next_state[0] - self.goal[0], next_state[1] - self.goal[1])
            else:
                curr_dist = new_dist = 0
            # The shaping reward is the reduction in distance (if agent gets closer, this is positive)
            shaping_reward = (curr_dist - new_dist)
            reward += shaping_reward
        return next_state, reward, done

    def render(self, path=None):
        """
        Visualize the grid world using matplotlib (static plot).
        - Obstacles are shown in black, free cells in white.
        - Start cell in green, Goal in red.
        - If a path (list of (r,c) states) is provided, those cells will be highlighted in blue.
        """
        import matplotlib.pyplot as plt
        # Prepare an RGB image for visualization
        img = np.ones((self.height, self.width, 3))  # start with white background for free cells
        # Mark obstacles as black
        obs_positions = np.where(self.grid == 1)
        img[obs_positions] = [0.0, 0.0, 0.0]
        # Mark start and goal
        sr, sc = self.start
        gr, gc = self.goal
        img[sr, sc] = [0.0, 1.0, 0.0]  # green start
        img[gr, gc] = [1.0, 0.0, 0.0]  # red goal
        # Mark path if provided
        if path:
            for (pr, pc) in path:
                if (pr, pc) != self.start and (pr, pc) != self.goal:
                    img[pr, pc] = [0.5, 0.5, 1.0]  # light blue for path
        # Plot the grid
        plt.figure(figsize=(5,5))
        plt.imshow(img, origin='upper')
        plt.title("GridWorld")
        plt.axis('off')
        plt.show()
